# EEG_10 — Temporal HGNN (T-HGNN)

Estensione di EEG_09: aggiunge un **encoder temporale 1D CNN per-nodo** prima del layer HGNN.

**Motivazione**: EEG_09 trattava i 384 timestep come feature scalari e li poolava globalmente.
Il segnale raw `x(61, 384)` contiene dinamiche temporali (onset, oscillazioni, transitori) ignorate
dall'HGNN statico → tutti i modelli a chance. T-HGNN encode prima la dinamica temporale
di ogni nodo, poi propaga lungo le iperedge.

**Architettura**:
```
x (B, N, T=384)
  → TemporalEncoder [Conv1D per-nodo, shared weights]
  → x_enc (B, N, d_temp=64)
  → HGNNConv × 2 layers
  → GlobalMeanPool → Linear → logits (B, 4)
```

Ablation: 5 metriche × **pruned only** = 5 configurazioni.
Split: TRAIN sogg 0-49, VAL 50-59, TEST 60-73.

**Baseline di confronto** (da EEG_09): HGNN statico pruned → test bAcc ~0.253–0.260 (chance).
**Grafi**: consensus-pruned — raw esclusi.
**Instance norm**: attiva (z-score per-canale per-trial, Bomatter 2024).

In [ ]:
import json, logging, re, time
from collections import defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.metrics import balanced_accuracy_score
from sklearn.utils.class_weight import compute_class_weight
from tqdm.auto import tqdm
import wandb

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)-8s %(message)s', datefmt='%H:%M:%S')
log = logging.getLogger('eeg10')

project_root = next((p for p in [Path.cwd()] + list(Path.cwd().parents) if (p / '.git').exists()), Path.cwd())
FIG_DIR = project_root / 'figures'; FIG_DIR.mkdir(exist_ok=True)

# ---- CONFIG ----
SFREQ          = 256
N_CHANNELS     = 61
N_SAMPLES      = 384
N_CLASSES      = 4
CLUSTER_SCHEME = 'concr4'

SUBJ_TRAIN = list(range(0, 50))
SUBJ_VAL   = list(range(50, 60))
SUBJ_TEST  = list(range(60, 74))

# Temporal encoder
D_TEMP         = 64     # dimensione embedding temporale per nodo

# HGNN
HIDDEN         = 128
N_LAYERS       = 2
DROPOUT        = 0.3

# Training
LR             = 1e-3
BATCH_SIZE     = 64
MAX_EPOCHS     = 60
PATIENCE       = 12
USE_INSTANCE_NORM = True
LABEL_SMOOTHING   = 0.1   # label smoothing → evita collasso su classe maggioritaria

METRICS  = ['pcc', 'abs_pcc', 'im_pcc', 'wpli', 'plv']
VARIANTS = [True]   # pruned only

WANDB_ENTITY  = 'uras-daniele22-politecnico-di-milano'
WANDB_PROJECT = 'miralis-imagined-speech'

_PAT = re.compile(r'^P(\d+)_S(\d+)$')
label2cluster = {int(k): int(v) for k, v in json.loads(
    (project_root / 'configs' / 'label_schemes' / 'labelid2cluster_concr4.json').read_text()).items()}

log.info(f'SUBJ_TRAIN={len(SUBJ_TRAIN)} VAL={len(SUBJ_VAL)} TEST={len(SUBJ_TEST)}')
log.info(f'D_TEMP={D_TEMP} HIDDEN={HIDDEN} N_LAYERS={N_LAYERS} label_smoothing={LABEL_SMOOTHING}')


## §2 — Dataset

In [ ]:
class HypergraphDataset(Dataset):
    """Carica PT ipergrafi da EEG_07f. Keys: H (61,E), x (61,384), y."""
    def __init__(self, subj_ids, metric, pruned, use_instance_norm=True):
        kind = 'hypergraphs_pruned' if pruned else 'hypergraphs'
        root = project_root / 'data' / f'{kind}_{metric}'
        self.paths, self.labels = [], []
        self.use_instance_norm = use_instance_norm
        for p in sorted(root.rglob('trial_*.pt')):
            m = _PAT.match(p.parent.name)
            if not m: continue
            sid = int(m.group(1))
            if sid not in subj_ids: continue
            d = torch.load(p, weights_only=False)
            y_word = int(d['y'].squeeze()) if isinstance(d['y'], torch.Tensor) else int(d['y'])
            c = label2cluster.get(y_word)
            if c is None: continue
            self.paths.append(p)
            self.labels.append(c)
        log.info(f'  {kind}_{metric}: {len(self.paths)} trial, {len(set(self.labels))} classi')

    def __len__(self): return len(self.paths)

    def __getitem__(self, idx):
        d = torch.load(self.paths[idx], weights_only=False)
        x = d['x'].float()   # (61, 384)
        H = d['H'].float()   # (61, E) — E variabile dopo pruning
        if self.use_instance_norm:
            x = (x - x.mean(dim=1, keepdim=True)) / (x.std(dim=1, keepdim=True) + 1e-6)
        # Pad/tronca H a (N_CHANNELS, N_CHANNELS)
        if H.shape[1] < N_CHANNELS:
            H = F.pad(H, (0, N_CHANNELS - H.shape[1]))
        elif H.shape[1] > N_CHANNELS:
            H = H[:, :N_CHANNELS]
        y = torch.tensor(self.labels[idx], dtype=torch.long)
        return x, H, y


def make_class_weights(dataset):
    """Weighted CrossEntropyLoss: peso inversamente proporzionale alla frequenza di classe."""
    labels = np.array(dataset.labels)
    weights = compute_class_weight('balanced', classes=np.arange(N_CLASSES), y=labels)
    log.info(f'  class weights: {np.round(weights, 3)}')
    return torch.tensor(weights, dtype=torch.float)


def make_sampler(dataset):
    """WeightedRandomSampler: bilancia le classi nei mini-batch."""
    labels = np.array(dataset.labels)
    class_counts = np.bincount(labels, minlength=N_CLASSES)
    sample_weights = 1.0 / class_counts[labels]   # peso per trial = 1 / freq_classe
    return WeightedRandomSampler(
        weights=torch.tensor(sample_weights, dtype=torch.float),
        num_samples=len(sample_weights),
        replacement=True,
    )


def make_loaders(metric, pruned):
    tr = HypergraphDataset(SUBJ_TRAIN, metric, pruned, USE_INSTANCE_NORM)
    va = HypergraphDataset(SUBJ_VAL,   metric, pruned, USE_INSTANCE_NORM)
    te = HypergraphDataset(SUBJ_TEST,  metric, pruned, USE_INSTANCE_NORM)
    kw = dict(num_workers=2, pin_memory=True)
    # Sampler bilanciato solo sul training set
    sampler = make_sampler(tr)
    tr_loader = DataLoader(tr, BATCH_SIZE, sampler=sampler, **kw)
    va_loader = DataLoader(va, BATCH_SIZE, shuffle=False, **kw)
    te_loader = DataLoader(te, BATCH_SIZE, shuffle=False, **kw)
    # Pesi per la loss
    class_weights = make_class_weights(tr)
    return tr_loader, va_loader, te_loader, class_weights


## §3 — Architettura T-HGNN

In [ ]:
# ---- HGNN conv layer (Feng et al. 2019) ----
class HGNNConv(nn.Module):
    """X' = Dv^{-1/2} H W De^{-1} H^T Dv^{-1/2} X Theta"""
    def __init__(self, in_ch, out_ch, bias=True):
        super().__init__()
        self.weight = nn.Parameter(torch.empty(in_ch, out_ch))
        self.bias   = nn.Parameter(torch.zeros(out_ch)) if bias else None
        nn.init.xavier_uniform_(self.weight)

    def forward(self, X, H):
        # X: (B, N, C_in)  H: (B, N, E)
        d_v = H.sum(dim=2).clamp(min=1e-6)             # (B, N)
        d_e = H.sum(dim=1).clamp(min=1e-6)             # (B, E)
        Dv_inv_sqrt = (1.0 / d_v.sqrt()).unsqueeze(-1)  # (B, N, 1)
        De_inv      = (1.0 / d_e).unsqueeze(1)          # (B, 1, E)
        XW  = X @ self.weight
        out = Dv_inv_sqrt * XW
        out = torch.bmm(H.transpose(1, 2), out)         # (B, E, C_out)
        out = De_inv.transpose(1, 2) * out
        out = torch.bmm(H, out)                          # (B, N, C_out)
        out = Dv_inv_sqrt * out
        if self.bias is not None:
            out = out + self.bias
        return out


# ---- Temporal encoder ----
class TemporalEncoder(nn.Module):
    """
    1D CNN per-nodo con pesi condivisi tra nodi.
    Input:  x (B, N, T)
    Output: x_enc (B, N, d_out)

    Architettura:
      Conv1d(1, 16, k=25, s=4)  → (B*N, 16, ~96)
      Conv1d(16, 32, k=15, s=4) → (B*N, 32, ~24)
      Conv1d(32, d_out, k=8, s=4) → (B*N, d_out, ~6)
      AdaptiveAvgPool1d(1) + Flatten → (B*N, d_out)
    """
    def __init__(self, T=N_SAMPLES, d_out=D_TEMP):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(1,   16,    kernel_size=25, stride=4, padding=12),
            nn.BatchNorm1d(16), nn.ELU(),
            nn.Conv1d(16,  32,    kernel_size=15, stride=4, padding=7),
            nn.BatchNorm1d(32), nn.ELU(),
            nn.Conv1d(32,  d_out, kernel_size=8,  stride=4, padding=3),
            nn.BatchNorm1d(d_out), nn.ELU(),
            nn.AdaptiveAvgPool1d(1),
            nn.Flatten(),   # (B*N, d_out)
        )

    def forward(self, x):
        B, N, T = x.shape
        x = x.reshape(B * N, 1, T)
        out = self.net(x)        # (B*N, d_out)
        return out.reshape(B, N, -1)  # (B, N, d_out)


# ---- T-HGNN ----
class TemporalHGNN(nn.Module):
    """
    TemporalEncoder → HGNNConv × n_layers → GlobalMeanPool → Linear
    """
    def __init__(self, T=N_SAMPLES, d_temp=D_TEMP, hidden=HIDDEN,
                 n_classes=N_CLASSES, n_layers=N_LAYERS, dropout=DROPOUT):
        super().__init__()
        self.temp_enc = TemporalEncoder(T, d_temp)
        dims = [d_temp] + [hidden] * n_layers
        self.convs = nn.ModuleList([HGNNConv(dims[i], dims[i + 1]) for i in range(n_layers)])
        self.bn    = nn.ModuleList([nn.BatchNorm1d(hidden) for _ in range(n_layers)])
        self.drop  = nn.Dropout(dropout)
        self.clf   = nn.Linear(hidden, n_classes)

    def forward(self, x, H):
        # x: (B, N, T)  H: (B, N, E)
        out = self.temp_enc(x)    # (B, N, d_temp)
        for conv, bn in zip(self.convs, self.bn):
            out = conv(out, H)    # (B, N, hidden)
            B, N, C = out.shape
            out = bn(out.reshape(B * N, C)).reshape(B, N, C)
            out = F.relu(out)
            out = self.drop(out)
        out = out.mean(dim=1)     # (B, hidden) — global mean pool
        return self.clf(out)      # (B, n_classes)


# Sanity check
_m = TemporalHGNN()
_x = torch.randn(4, N_CHANNELS, N_SAMPLES)
_H = torch.rand(4, N_CHANNELS, N_CHANNELS)
assert _m(_x, _H).shape == (4, N_CLASSES), 'forward check FAIL'
n_params = sum(p.numel() for p in _m.parameters() if p.requires_grad)
log.info(f'TemporalHGNN OK — {n_params:,} parametri')
del _m, _x, _H


## §4 — Train / Eval

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
log.info(f'device: {device}')


def run_epoch(model, loader, optimizer=None, criterion=None):
    train = optimizer is not None
    model.train() if train else model.eval()
    total_loss, all_labels, all_preds = 0.0, [], []
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for x, H, y in loader:
            x, H, y = x.to(device), H.to(device), y.to(device)
            logits = model(x, H)
            loss = criterion(logits, y)
            if train:
                optimizer.zero_grad(); loss.backward(); optimizer.step()
            total_loss += loss.item() * len(y)
            all_labels.extend(y.cpu().numpy())
            all_preds.extend(logits.argmax(1).cpu().numpy())
    bacc = balanced_accuracy_score(all_labels, all_preds)
    return total_loss / len(loader.dataset), bacc, np.array(all_labels), np.array(all_preds)


def train_model(run_name, tr_loader, va_loader, te_loader, class_weights, config):
    run = wandb.init(entity=WANDB_ENTITY, project=WANDB_PROJECT,
                     name=run_name, config=config, reinit='finish_previous',
                     settings=wandb.Settings(start_method='thread'))

    # Triplice protezione contro class collapse:
    # 1) WeightedRandomSampler → batch bilanciati (nel DataLoader)
    # 2) class_weights nella loss → penalizza errori su classi rare
    # 3) label_smoothing → evita predizioni troppo sicure su una sola classe
    criterion = nn.CrossEntropyLoss(
        weight=class_weights.to(device),
        label_smoothing=LABEL_SMOOTHING,
    )

    model = TemporalHGNN().to(device)
    opt   = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=MAX_EPOCHS)
    best_val, best_state, patience_cnt = 0.0, None, 0

    for epoch in range(1, MAX_EPOCHS + 1):
        tr_loss, tr_bacc, _, _ = run_epoch(model, tr_loader, opt, criterion)
        va_loss, va_bacc, _, _ = run_epoch(model, va_loader, criterion=criterion)
        sched.step()
        run.log({'train/loss': tr_loss, 'train/bacc': tr_bacc,
                 'val/loss': va_loss,   'val/bacc': va_bacc, 'epoch': epoch})
        if va_bacc > best_val:
            best_val = va_bacc
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_cnt = 0
        else:
            patience_cnt += 1
        if patience_cnt >= PATIENCE:
            log.info(f'  early stop epoch {epoch}'); break

    model.load_state_dict(best_state)
    _, te_bacc, te_labels, te_preds = run_epoch(model, te_loader, criterion=criterion)
    run.summary['val_bacc']  = best_val
    run.summary['test_bacc'] = te_bacc
    run.log({'confusion_matrix': wandb.plot.confusion_matrix(
        preds=te_preds.tolist(), y_true=te_labels.tolist(),
        class_names=['CONCR', 'AZIONE', 'STATO', 'ASTRATTO'])})
    run.finish()
    log.info(f'  {run_name}: val={best_val:.4f} test={te_bacc:.4f}')
    return best_val, te_bacc, te_labels, te_preds


## §5 — Ablation Loop (5 config — pruned only)

In [ ]:
RESULTS = {}  # key: '{metric}_pruned' -> {'val': float, 'test': float}

for metric in METRICS:
    for pruned in VARIANTS:
        key  = f'{metric}_{"pruned" if pruned else "raw"}'
        name = f'eeg10_THGNN_{key}_{CLUSTER_SCHEME}'
        log.info(f'\n=== {name} ===')

        try:
            tr_l, va_l, te_l, cw = make_loaders(metric, pruned)
        except Exception as e:
            log.warning(f'  Skip {key}: {e}'); continue

        cfg = dict(
            notebook='EEG_10', model='TemporalHGNN', metric=metric, pruned=pruned,
            n_classes=N_CLASSES, cluster_scheme=CLUSTER_SCHEME,
            d_temp=D_TEMP, hidden=HIDDEN, n_layers=N_LAYERS, dropout=DROPOUT,
            lr=LR, batch_size=BATCH_SIZE, max_epochs=MAX_EPOCHS,
            use_instance_norm=USE_INSTANCE_NORM,
            label_smoothing=LABEL_SMOOTHING,
            weighted_sampler=True,
            n_train_subj=len(SUBJ_TRAIN),
        )

        val_b, test_b, lbl, pred = train_model(name, tr_l, va_l, te_l, cw, cfg)
        RESULTS[key] = {'val': val_b, 'test': test_b}

log.info('\n=== ABLATION DONE ===')
for k, v in RESULTS.items():
    print(f'  {k:30s}  val={v["val"]:.4f}  test={v["test"]:.4f}')


## §6 — Risultati + Confronto con EEG_09

In [ ]:
# Baseline EEG_09 (HGNN statico pruned) per confronto
EEG09_BASELINE = {
    'pcc_pruned':     {'val': 0.264, 'test': 0.260},
    'abs_pcc_pruned': {'val': 0.262, 'test': 0.253},
    'im_pcc_pruned':  {'val': 0.260, 'test': 0.256},
    'wpli_pruned':    {'val': 0.261, 'test': 0.255},
    'plv_pruned':     {'val': 0.255, 'test': 0.260},
}

if RESULTS:
    var_labels = ['pruned' if v else 'raw' for v in VARIANTS]
    chance = 1 / N_CLASSES

    # ---- Bar chart T-HGNN ----
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    fig.suptitle('EEG_10 — T-HGNN Ablation (bAcc) [pruned]', fontsize=13, fontweight='bold')

    for ax, split in zip(axes, ['val', 'test']):
        vals10 = [RESULTS.get(f'{m}_pruned', {}).get(split, np.nan) for m in METRICS]
        vals09 = [EEG09_BASELINE.get(f'{m}_pruned', {}).get(split, np.nan) for m in METRICS]
        y_pos  = np.arange(len(METRICS))
        h = 0.35

        bars10 = ax.barh(y_pos + h/2, vals10, height=h, label='T-HGNN (EEG_10)',
                         color='#1976D2', edgecolor='white')
        bars09 = ax.barh(y_pos - h/2, vals09, height=h, label='HGNN static (EEG_09)',
                         color='#90CAF9', edgecolor='white')
        ax.axvline(chance, color='gray', linestyle='--', linewidth=1.2,
                   label=f'Chance ({chance:.2f})')
        ax.set_yticks(y_pos); ax.set_yticklabels(METRICS, fontsize=10)
        ax.set_xlim(0.18, max(0.50, max([v for v in vals10 if not np.isnan(v)] or [0.5]) + 0.06))
        ax.set_xlabel('Balanced Accuracy')
        ax.set_title(f'{split.capitalize()} bAcc', fontsize=11)
        ax.legend(fontsize=8, loc='lower right')

        for bar, v in zip(bars10, vals10):
            if not np.isnan(v):
                ax.text(v + 0.003, bar.get_y() + bar.get_height() / 2,
                        f'{v:.3f}', va='center', fontsize=8)

    plt.tight_layout()
    plt.savefig(FIG_DIR / 'eeg10_thgnn_ablation.png', dpi=150, bbox_inches='tight')
    plt.show()

    # ---- Delta vs EEG_09 ----
    print('\n--- Delta T-HGNN vs HGNN statico (EEG_09 baseline) ---')
    rows = []
    for m in METRICS:
        k = f'{m}_pruned'
        v10_test = RESULTS.get(k, {}).get('test', np.nan)
        v09_test = EEG09_BASELINE.get(k, {}).get('test', np.nan)
        delta    = v10_test - v09_test if not np.isnan(v10_test) else np.nan
        rows.append({'metric': m, 'eeg09_test': v09_test,
                     'eeg10_test': v10_test, 'delta': delta})
    df = pd.DataFrame(rows)
    print(df.to_string(index=False))

    df_full = pd.DataFrame([
        {'metric': m, 'variant': vl,
         'val_bacc':  RESULTS.get(f'{m}_{vl}', {}).get('val',  np.nan),
         'test_bacc': RESULTS.get(f'{m}_{vl}', {}).get('test', np.nan)}
        for m in METRICS for vl in var_labels
    ])
    df_full.to_csv(FIG_DIR / 'eeg10_thgnn_results.csv', index=False)
